In [2]:
# =============================================================================
# GOLD — Délinquance (vols liés aux véhicules, dégradations) + carte HTML
# Notebook Microsoft Fabric (PySpark). Lakehouse attaché au notebook.
#
# Entrées  : silver_delinquance_departement, silver_delinquance_commune
# Sorties  : gold_delinquance_region / _departement / _commune  (tables Delta)
#            Files/gold/delinquance/carte_delinquance.html     (carte interactive)
#
# Collez chaque bloc "# %%" dans une cellule séparée du notebook.
# =============================================================================


# %% [Cellule 1] — Tables Gold ------------------------------------------------
from pyspark.sql import functions as F

VEH = ["vol_vehicule", "vol_dans_vehicule", "vol_accessoires_vehicule"]

def with_taux(df):
    # même définition à tous les niveaux : faits pour 1 000 habitants
    return df.withColumn(
        "taux_pour_mille",
        F.when(F.col("population") > 0,
               F.round(F.col("nombre") / F.col("population") * 1000, 3)))

# ---------- Département ----------
dep = spark.table("silver_delinquance_departement") \
    .select("code_departement", "code_region", "annee", "indicateur_code", "nombre", "population")

dep_total_veh = (dep.filter(F.col("indicateur_code").isin(VEH))
    .groupBy("code_departement", "code_region", "annee")
    .agg(F.sum("nombre").alias("nombre"), F.max("population").alias("population"))
    .withColumn("indicateur_code", F.lit("vols_vehicules_total")))

gold_dep = with_taux(dep.unionByName(dep_total_veh))
gold_dep.write.mode("overwrite").format("delta").option("overwriteSchema", "true") \
    .saveAsTable("gold_delinquance_departement")

# ---------- Région (agrégée depuis les départements) ----------
gold_reg = with_taux(
    gold_dep.groupBy("code_region", "annee", "indicateur_code")
            .agg(F.sum("nombre").alias("nombre"), F.sum("population").alias("population")))
gold_reg.write.mode("overwrite").format("delta").option("overwriteSchema", "true") \
    .saveAsTable("gold_delinquance_region")

# ---------- Commune ----------
com = spark.table("silver_delinquance_commune") \
    .select("code_commune", "code_departement", "annee", "indicateur_code",
            "nombre", "population", "est_diffuse")

# Paris / Lyon / Marseille : la base donne les arrondissements.
# On reconstruit la commune (75056, 69123, 13055) si elle est absente.
parent = (F.when(F.col("code_commune").between("75101", "75120"), F.lit("75056"))
           .when(F.col("code_commune").between("69381", "69389"), F.lit("69123"))
           .when(F.col("code_commune").between("13201", "13216"), F.lit("13055")))
plm = (com.withColumn("parent", parent).filter(F.col("parent").isNotNull())
    .groupBy("parent", "code_departement", "annee", "indicateur_code")
    .agg(F.sum("nombre").alias("nombre"), F.sum("population").alias("population"),
         F.min(F.col("est_diffuse").cast("int")).cast("boolean").alias("est_diffuse"))
    .withColumnRenamed("parent", "code_commune")
    .withColumn("nombre", F.when(F.col("est_diffuse"), F.col("nombre"))))
plm = plm.join(com.select("code_commune").distinct(), "code_commune", "left_anti")
com = com.unionByName(plm.select(com.columns))

# Total des vols liés aux véhicules : seulement si les 3 indicateurs sont diffusés
com_total_veh = (com.filter(F.col("indicateur_code").isin(VEH))
    .groupBy("code_commune", "code_departement", "annee")
    .agg(F.sum("nombre").alias("s"),
         F.sum(F.col("est_diffuse").cast("int")).alias("nb_diff"),
         F.max("population").alias("population"))
    .withColumn("est_diffuse", F.col("nb_diff") == 3)
    .withColumn("nombre", F.when(F.col("est_diffuse"), F.col("s")))
    .withColumn("indicateur_code", F.lit("vols_vehicules_total"))
    .select(com.columns))

gold_com = with_taux(com.unionByName(com_total_veh))
gold_com.write.mode("overwrite").format("delta").option("overwriteSchema", "true") \
    .partitionBy("annee").saveAsTable("gold_delinquance_commune")

for t in ["gold_delinquance_region", "gold_delinquance_departement", "gold_delinquance_commune"]:
    print(t, spark.table(t).count(), "lignes")


# %% [Cellule 2] — Carte HTML -------------------------------------------------
import json, os, requests
import pandas as pd

OUT_DIR = "/lakehouse/default/Files/gold/delinquance"
GEO_BASE = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/"

# --- TEMPLATE START ---
INDICATEURS = [
    {"code": "vols_vehicules_total", "label": "Vols liés aux véhicules (total)", "unit": "véhicules"},
    {"code": "vol_vehicule", "label": "Vols de véhicules", "unit": "véhicules"},
    {"code": "vol_dans_vehicule", "label": "Vols dans les véhicules", "unit": "véhicules"},
    {"code": "vol_accessoires_vehicule", "label": "Vols d'accessoires sur véhicules", "unit": "véhicules"},
    {"code": "degradation", "label": "Destructions et dégradations volontaires", "unit": "infractions"},
]

def _num(x):
    if x is None or pd.isna(x):
        return None
    return int(round(float(x)))

def pack(pdf, code_col, years):
    """{code: {"p": [pop par année], "n": {indicateur: [nombre par année]}}}"""
    out = {}
    pdf = pdf.copy()
    pdf[code_col] = pdf[code_col].astype(str)
    pop = pdf.groupby([code_col, "annee"])["population"].max().unstack("annee").reindex(columns=years)
    for code, row in pop.iterrows():
        out[code] = {"p": [_num(v) for v in row.tolist()], "n": {}}
    for ind, g in pdf.groupby("indicateur_code"):
        n = g.groupby([code_col, "annee"])["nombre"].sum(min_count=1).unstack("annee").reindex(columns=years)
        for code, row in n.iterrows():
            vals = [_num(v) for v in row.tolist()]
            if any(v is not None for v in vals):
                out[code]["n"][ind] = vals
    return out

def _round_coords(c, nd=3):
    if isinstance(c[0], (int, float)):
        return [round(c[0], nd), round(c[1], nd)]
    pts = [_round_coords(x, nd) for x in c]
    if pts and isinstance(pts[0][0], (int, float)):          # anneau
        ded = [pts[0]] + [p for i, p in enumerate(pts[1:], 1) if p != pts[i - 1]]
        return ded if len(ded) >= 4 else pts
    return pts

def slim_geojson(gj, nd=3):
    for f in gj["features"]:
        f["properties"] = {"code": f["properties"]["code"], "nom": f["properties"]["nom"]}
        f["geometry"]["coordinates"] = _round_coords(f["geometry"]["coordinates"], nd)
    return gj

def build_html(data, reg_geo, dep_geo):
    payload = json.dumps({"data": data, "reg": reg_geo, "dep": dep_geo},
                         ensure_ascii=False, separators=(",", ":"))
    return HTML_TEMPLATE.replace("__PAYLOAD__", payload.replace("</", "<\\/"))

HTML_TEMPLATE = r"""<!doctype html>
<html lang="fr">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Carte de la délinquance</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css">
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>
:root{
  --bg:#f7f6f3; --surface:#ffffff; --ink:#1f1e1c; --ink-2:#55534e; --muted:#8a877f;
  --line:#e4e2dc; --accent:#256abf; --nodata:#d9d7d1; --hl:#1f1e1c;
  --c0:#cde2fb; --c1:#9ec5f4; --c2:#6da7ec; --c3:#3987e5; --c4:#256abf; --c5:#104281;
}
@media (prefers-color-scheme: dark){
  :root:not([data-theme="light"]){
    --bg:#121211; --surface:#1a1a19; --ink:#f1efe9; --ink-2:#bdbab2; --muted:#8a877f;
    --line:#2e2d2a; --accent:#6da7ec; --nodata:#3a3936; --hl:#ffffff;
  }
}
*{box-sizing:border-box}
html,body{margin:0;height:100%}
body{background:var(--bg);color:var(--ink);font:14px/1.45 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif;
  display:grid;grid-template-columns:minmax(0,1fr) 360px;grid-template-rows:auto minmax(0,1fr);height:100vh}
header{grid-column:1/-1;display:flex;flex-wrap:wrap;gap:10px 16px;align-items:center;padding:10px 16px;
  background:var(--surface);border-bottom:1px solid var(--line)}
header h1{font-size:16px;margin:0 8px 0 0;font-weight:650;letter-spacing:-.01em}
label.ctl{display:flex;flex-direction:column;font-size:11px;color:var(--muted);gap:2px}
select,button{font:inherit;color:var(--ink);background:var(--bg);border:1px solid var(--line);border-radius:6px;padding:5px 8px}
button{cursor:pointer} button:hover{border-color:var(--accent)}
.seg{display:inline-flex;border:1px solid var(--line);border-radius:6px;overflow:hidden}
.seg button{border:0;border-radius:0;background:transparent}
.seg button[aria-pressed="true"]{background:var(--accent);color:#fff}
#yearOut{font-variant-numeric:tabular-nums;font-weight:600;color:var(--ink);font-size:13px}
#map{min-height:0;background:var(--bg)}
aside{background:var(--surface);border-left:1px solid var(--line);overflow:auto;padding:14px 16px}
.crumb{font-size:12px;color:var(--ink-2);margin-bottom:8px}
.crumb a{color:var(--accent);cursor:pointer;text-decoration:none}
.kpi{font-size:28px;font-weight:650;font-variant-numeric:tabular-nums;letter-spacing:-.02em}
.kpi small{font-size:13px;font-weight:400;color:var(--ink-2)}
.sub{color:var(--ink-2);font-size:12.5px}
h2{font-size:12px;text-transform:uppercase;letter-spacing:.06em;color:var(--muted);margin:18px 0 6px;font-weight:600}
#legend .row{display:flex;align-items:center;gap:8px;font-size:12px;font-variant-numeric:tabular-nums;color:var(--ink-2);margin:2px 0}
#legend .sw{width:22px;height:12px;border-radius:3px;flex:none}
.hatch{background:repeating-linear-gradient(45deg,var(--nodata) 0 3px,transparent 3px 6px);border:1px solid var(--nodata)}
table{width:100%;border-collapse:collapse;font-size:12.5px;font-variant-numeric:tabular-nums}
td,th{padding:4px 2px;border-bottom:1px solid var(--line);text-align:left}
th{color:var(--muted);font-weight:500;font-size:11px}
td.r,th.r{text-align:right}
tbody tr{cursor:pointer} tbody tr:hover{background:var(--bg)}
#spark{width:100%;height:90px;display:block}
#spark text{fill:var(--muted);font-size:10px}
.note{font-size:11px;color:var(--muted);margin-top:16px}
#msg{position:absolute;left:50%;top:70px;transform:translateX(-50%);z-index:1000;background:var(--surface);color:var(--ink);
  border:1px solid var(--line);padding:6px 12px;border-radius:6px;font-size:12px;display:none}
.leaflet-container{font:inherit}
@media (prefers-color-scheme: dark){ .osm-tiles{filter:invert(1) hue-rotate(180deg) brightness(.85) contrast(.9)} }
.leaflet-tooltip.tt{background:var(--surface);color:var(--ink);border:1px solid var(--line);border-radius:6px;
  box-shadow:0 2px 8px rgba(0,0,0,.12);padding:6px 9px;font-size:12px}
.tt b{font-size:13px}
.drom{display:flex;gap:4px;flex-wrap:wrap}
.drom button{font-size:11px;padding:3px 6px}
@media (max-width:820px){
  body{grid-template-columns:1fr;grid-template-rows:auto 60vh auto;height:auto}
  aside{border-left:0;border-top:1px solid var(--line)}
}
</style>
</head>
<body>
<header>
  <h1>Délinquance enregistrée</h1>
  <label class="ctl">Niveau
    <span class="seg" id="lvl">
      <button data-v="reg" aria-pressed="true">Régions</button>
      <button data-v="dep" aria-pressed="false">Départements</button>
      <button data-v="com" aria-pressed="false">Communes</button>
    </span>
  </label>
  <label class="ctl">Indicateur <select id="ind"></select></label>
  <label class="ctl">Mesure
    <span class="seg" id="met">
      <button data-v="taux" aria-pressed="true">Pour 1 000 hab.</button>
      <button data-v="nombre" aria-pressed="false">Nombre</button>
    </span>
  </label>
  <label class="ctl">Année <span style="display:flex;gap:8px;align-items:center">
    <input type="range" id="year"><span id="yearOut"></span></span></label>
  <label class="ctl">Outre-mer <span class="drom" id="drom"></span></label>
</header>
<div id="map"></div>
<aside>
  <div class="crumb" id="crumb"></div>
  <div id="title" style="font-weight:600;font-size:15px"></div>
  <div class="kpi" id="kpi"></div>
  <div class="sub" id="kpiSub"></div>
  <h2>Évolution 2016 → dernière année</h2>
  <svg id="spark" role="img"></svg>
  <h2>Légende</h2>
  <div id="legend"></div>
  <h2 id="topTitle">Classement</h2>
  <table><thead><tr><th>#</th><th>Territoire</th><th class="r" id="thVal">Valeur</th></tr></thead><tbody id="top"></tbody></table>
  <p class="note">Source : SSMSI – ministère de l'Intérieur, délinquance enregistrée par la police et la gendarmerie (lieu de commission), Licence Ouverte 2.0.
  Taux = faits pour 1 000 habitants. Communes : valeurs non diffusées sous un seuil (secret statistique), affichées en gris.
  Les vols se comptent en véhicules, les dégradations en infractions : ne pas les additionner.</p>
</aside>
<div id="msg"></div>
<script id="payload" type="application/json">__PAYLOAD__</script>
<script>
(function(){
const P = JSON.parse(document.getElementById('payload').textContent);
const D = P.data, YEARS = D.years, INDS = D.indicators;
const REG = D.reg, DEP = D.dep, COM = D.com;
const DEP_REG = D.dep_region;
const nf = new Intl.NumberFormat('fr-FR');
const nf2 = new Intl.NumberFormat('fr-FR',{maximumFractionDigits:2, minimumFractionDigits:2});
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const CLASSES = ['--c0','--c1','--c2','--c3','--c4','--c5'];

let breaks = [];
const S = {lvl:'reg', ind:INDS[0].code, met:'taux', yi:YEARS.length-1, dep:null, reg:null, sel:null};
const regName = {}, depName = {};
P.reg.features.forEach(f=>regName[f.properties.code]=f.properties.nom);
P.dep.features.forEach(f=>depName[f.properties.code]=f.properties.nom);
const comName = {};

// ---------- contrôles ----------
const indSel = document.getElementById('ind');
INDS.forEach(i=>{const o=document.createElement('option');o.value=i.code;o.textContent=i.label;indSel.appendChild(o);});
indSel.onchange = ()=>{S.ind=indSel.value; render();};
const yr = document.getElementById('year');
yr.min=0; yr.max=YEARS.length-1; yr.value=S.yi;
yr.oninput = ()=>{S.yi=+yr.value; render();};
function seg(id,key,cb){
  const el=document.getElementById(id);
  el.querySelectorAll('button').forEach(b=>b.onclick=()=>{
    el.querySelectorAll('button').forEach(x=>x.setAttribute('aria-pressed', x===b));
    cb(b.dataset.v);
  });
  return v=>el.querySelectorAll('button').forEach(x=>x.setAttribute('aria-pressed', x.dataset.v===v));
}
const setLvlBtn = seg('lvl','lvl', v=>{
  if(v==='com' && !S.dep){ flash('Cliquez d’abord sur un département pour afficher ses communes.'); setLvlBtn(S.lvl); return; }
  goLevel(v);
});
seg('met','met', v=>{S.met=v; render();});

// ---------- carte ----------
const map = L.map('map',{zoomSnap:0.25, preferCanvas:false}).setView([46.6,2.4],5.75);
L.tileLayer('https://tile.openstreetmap.org/{z}/{x}/{y}.png',{
  attribution:'&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors',
  maxZoom:19, className:'osm-tiles'}).addTo(map);
const FR_BOUNDS = [[41.2,-5.3],[51.2,9.7]];
map.fitBounds(FR_BOUNDS);

const DROM = {'971':'Guadeloupe','972':'Martinique','973':'Guyane','974':'Réunion','976':'Mayotte'};
const dromEl = document.getElementById('drom');
const mb=document.createElement('button'); mb.textContent='Métropole'; mb.onclick=()=>map.fitBounds(FR_BOUNDS); dromEl.appendChild(mb);
Object.entries(DROM).forEach(([c,n])=>{const b=document.createElement('button');b.textContent=n;
  b.onclick=()=>{const l=findLayer(depLayer,c); if(l) map.fitBounds(l.getBounds(),{padding:[20,20]});}; dromEl.appendChild(b);});

function findLayer(layer,code){let r=null; layer && layer.eachLayer(l=>{if(l.feature.properties.code===code) r=l;}); return r;}

const regLayer = L.geoJSON(P.reg,{style:styleFor(()=>REG), onEachFeature:bind('reg')});
const depLayer = L.geoJSON(P.dep,{style:styleFor(()=>DEP), onEachFeature:bind('dep')});
let comLayer = null;
const comCache = {};

function source(lvl){return lvl==='reg'?REG:lvl==='dep'?DEP:COM;}
function nameOf(lvl,code){return lvl==='reg'?regName[code]:lvl==='dep'?depName[code]:(comName[code]||code);}

function val(e,ind,yi,met){
  if(!e) return null;
  const n = e.n[ind] ? e.n[ind][yi] : null;
  if(n==null) return null;
  if(met==='nombre') return n;
  const p=e.p[yi]; return p? n/p*1000 : null;
}
function fmt(v,met){ if(v==null) return 'n.d.'; return met==='nombre'? nf.format(Math.round(v)) : nf2.format(v); }

// classes par quantiles sur les territoires affichés
function computeBreaks(codes, src){
  const vs = codes.map(c=>val(src[c],S.ind,S.yi,S.met)).filter(v=>v!=null).sort((a,b)=>a-b);
  if(!vs.length){breaks=[];return;}
  const b=[]; for(let k=1;k<CLASSES.length;k++){ b.push(vs[Math.min(vs.length-1,Math.floor(k*vs.length/CLASSES.length))]); }
  breaks = [...new Set(b)];
}
function colorOf(v){
  if(v==null) return null;
  let k=0; while(k<breaks.length && v>=breaks[k]) k++;
  const idx = breaks.length<CLASSES.length-1 ? Math.round(k*(CLASSES.length-1)/Math.max(1,breaks.length)) : k;
  return css(CLASSES[Math.min(idx,CLASSES.length-1)]);
}
function styleFor(getSrc){
  return f=>{
    const v=val(getSrc()[f.properties.code],S.ind,S.yi,S.met), c=colorOf(v);
    return {weight:0.8, color:css('--surface'), opacity:1, fillOpacity: c?0.85:0.55,
            fillColor: c || css('--nodata'), dashArray:null};
  };
}

function bind(lvl){
  return (f,layer)=>{
    layer.on('mouseover',e=>{ e.target.setStyle({weight:2.5,color:css('--hl')}); e.target.bringToFront(); showInfo(lvl,f.properties.code); });
    layer.on('mouseout',e=>{ currentLayer().resetStyle(e.target); showInfo(); });
    layer.on('click',()=>drill(lvl,f.properties.code,layer));
    layer.bindTooltip(()=>{
      const code=f.properties.code, e=source(lvl)[code];
      const t=val(e,S.ind,S.yi,'taux'), n=val(e,S.ind,S.yi,'nombre');
      return `<b>${nameOf(lvl,code)}</b><br>${fmt(t,'taux')} pour 1 000 hab.<br>${fmt(n,'nombre')} ${unit()} en ${YEARS[S.yi]}`;
    },{sticky:true, className:'tt', direction:'top', offset:[0,-8]});
  };
}
function unit(){return INDS.find(i=>i.code===S.ind).unit;}
function currentLayer(){return S.lvl==='reg'?regLayer:S.lvl==='dep'?depLayer:comLayer;}

function drill(lvl,code,layer){
  if(lvl==='reg'){ S.reg=code; map.fitBounds(layer.getBounds(),{padding:[20,20]}); goLevel('dep'); }
  else if(lvl==='dep'){ S.dep=code; S.reg=DEP_REG[code]||S.reg; loadCommunes(code, layer.getBounds()); }
  else { S.sel = code; showInfo('com',code); }
}

function goLevel(lvl){
  [regLayer,depLayer,comLayer].forEach(l=>l && map.hasLayer(l) && map.removeLayer(l));
  S.lvl=lvl; setLvlBtn(lvl);
  if(lvl==='reg'){ S.reg=null; S.dep=null; regLayer.addTo(map); map.fitBounds(FR_BOUNDS); }
  if(lvl==='dep'){ S.dep=null; depLayer.addTo(map); }
  if(lvl==='com'){ comLayer.addTo(map); }
  render();
}

async function loadCommunes(dep,bounds){
  flash('Chargement des communes…', true);
  try{
    if(!comCache[dep]){
      const url=`https://geo.api.gouv.fr/departements/${dep}/communes?fields=nom,code&format=geojson&geometry=contour`;
      const r=await fetch(url); if(!r.ok) throw new Error(r.status);
      comCache[dep]=await r.json();
    }
    const gj=comCache[dep];
    gj.features.forEach(f=>comName[f.properties.code]=f.properties.nom);
    if(comLayer) map.removeLayer(comLayer);
    comLayer = L.geoJSON(gj,{style:styleFor(()=>COM), onEachFeature:bind('com')});
    goLevel('com');
    map.fitBounds(bounds||comLayer.getBounds(),{padding:[20,20]});
    flash();
  }catch(err){
    flash('Impossible de charger les contours des communes (geo.api.gouv.fr). Vérifiez la connexion.');
  }
}

// ---------- panneau ----------
function allCodes(){
  if(S.lvl==='reg') return Object.keys(regName);
  if(S.lvl==='dep') return Object.keys(depName);
  return displayed();
}
function displayed(){
  if(S.lvl==='reg') return Object.keys(regName);
  if(S.lvl==='dep') return Object.keys(depName).filter(c=>!S.reg || DEP_REG[c]===S.reg);
  return comLayer ? comLayer.getLayers().map(l=>l.feature.properties.code) : [];
}
function franceEntity(){
  const e={p:YEARS.map(()=>0), n:{}};
  Object.values(REG).forEach(r=>{
    r.p.forEach((v,i)=>e.p[i]+= v||0);
    Object.entries(r.n).forEach(([k,arr])=>{ e.n[k]=e.n[k]||YEARS.map(()=>0); arr.forEach((v,i)=>e.n[k][i]+= v||0); });
  });
  return e;
}
const FRANCE = franceEntity();

function contextEntity(){
  if(S.lvl==='com' && S.dep) return ['dep',S.dep,DEP[S.dep]];
  if(S.lvl==='dep' && S.reg) return ['reg',S.reg,REG[S.reg]];
  return ['fr','France',FRANCE];
}
function showInfo(lvl,code){
  let e,label;
  if(lvl){ e=source(lvl)[code]; label=nameOf(lvl,code); }
  else if(S.sel && S.lvl==='com'){ e=COM[S.sel]; label=nameOf('com',S.sel); }
  else { const c=contextEntity(); e=c[2]; label=c[0]==='fr'?'France entière':nameOf(c[0],c[1]); }
  const ind=INDS.find(i=>i.code===S.ind);
  document.getElementById('title').textContent = label;
  const t=val(e,S.ind,S.yi,'taux'), n=val(e,S.ind,S.yi,'nombre');
  document.getElementById('kpi').innerHTML = S.met==='taux'
    ? `${fmt(t,'taux')} <small>pour 1 000 hab.</small>` : `${fmt(n,'nombre')} <small>${ind.unit}</small>`;
  const pop = e && e.p[S.yi];
  document.getElementById('kpiSub').textContent =
    `${ind.label} · ${YEARS[S.yi]} · ` + (S.met==='taux' ? `${fmt(n,'nombre')} ${ind.unit}` : `${fmt(t,'taux')} pour 1 000 hab.`)
    + (pop? ` · ${nf.format(pop)} hab.`:'');
  spark(e);
}
function spark(e){
  const svg=document.getElementById('spark'), W=svg.clientWidth||320, H=90, pad=14;
  const vs=YEARS.map((y,i)=>val(e,S.ind,i,S.met));
  const max=Math.max(...vs.filter(v=>v!=null),0)||1, bw=(W-4)/YEARS.length;
  let s='';
  vs.forEach((v,i)=>{
    const x=2+i*bw, h=v==null?0:Math.max(1,(H-pad-12)*v/max), y=H-pad-h;
    const fill = i===S.yi ? css('--c5') : css('--c2');
    s+= v==null ? `<rect x="${x+bw*0.15}" y="${H-pad-3}" width="${bw*0.7}" height="3" fill="${css('--nodata')}"/>`
               : `<rect x="${x+bw*0.15}" y="${y}" width="${bw*0.7}" height="${h}" rx="2" fill="${fill}"><title>${YEARS[i]} : ${fmt(v,S.met)}</title></rect>`;
    if(i===0||i===YEARS.length-1||i===S.yi) s+=`<text x="${x+bw/2}" y="${H-2}" text-anchor="middle">${YEARS[i]}</text>`;
  });
  if(vs[S.yi]!=null){ const x=2+S.yi*bw+bw/2; s+=`<text x="${x}" y="10" text-anchor="middle" style="fill:var(--ink)">${fmt(vs[S.yi],S.met)}</text>`; }
  svg.setAttribute('viewBox',`0 0 ${W} ${H}`); svg.innerHTML=s;
}
function legend(){
  const el=document.getElementById('legend'); let h='';
  const k=breaks.length+1;
  for(let i=0;i<k;i++){
    const lo = i===0 ? null : breaks[i-1], hi = i<breaks.length ? breaks[i] : null;
    const sample = lo==null ? (hi!=null? hi-1e-9 : 0) : lo;
    const txt = lo==null ? `moins de ${fmt(hi,S.met)}` : hi==null ? `${fmt(lo,S.met)} et plus` : `${fmt(lo,S.met)} – ${fmt(hi,S.met)}`;
    h+=`<div class="row"><span class="sw" style="background:${colorOf(sample)}"></span>${breaks.length?txt:''}</div>`;
  }
  h+=`<div class="row"><span class="sw" style="background:var(--nodata)"></span>Non diffusé / pas de donnée</div>`;
  h+=`<div class="row" style="color:var(--muted)">Classes par quantiles (${CLASSES.length}) des territoires affichés</div>`;
  el.innerHTML=h;
}
function top(){
  const src=source(S.lvl), codes=displayed();
  const rows=codes.map(c=>[c,val(src[c],S.ind,S.yi,S.met)]).filter(r=>r[1]!=null).sort((a,b)=>b[1]-a[1]).slice(0,15);
  const lvlTxt={reg:'régions',dep:'départements',com:'communes'}[S.lvl];
  document.getElementById('topTitle').textContent=`Top ${rows.length} ${lvlTxt}`;
  document.getElementById('thVal').textContent = S.met==='taux'?'‰ hab.':'Nombre';
  const tb=document.getElementById('top'); tb.innerHTML='';
  rows.forEach(([c,v],i)=>{
    const tr=document.createElement('tr');
    tr.innerHTML=`<td>${i+1}</td><td>${nameOf(S.lvl,c)}</td><td class="r">${fmt(v,S.met)}</td>`;
    tr.onclick=()=>{const l=findLayer(currentLayer(),c); if(l){ map.fitBounds(l.getBounds(),{padding:[40,40],maxZoom:11}); l.openTooltip(l.getBounds().getCenter()); }};
    tr.onmouseenter=()=>showInfo(S.lvl,c); tr.onmouseleave=()=>showInfo();
    tb.appendChild(tr);
  });
}
function crumb(){
  const el=document.getElementById('crumb'); const parts=[`<a data-go="fr">France</a>`];
  if(S.reg) parts.push(`<a data-go="reg">${regName[S.reg]}</a>`);
  if(S.dep) parts.push(`<span>${depName[S.dep]}</span>`);
  el.innerHTML=parts.join(' › ');
  el.querySelectorAll('a').forEach(a=>a.onclick=()=>{
    if(a.dataset.go==='fr') goLevel('reg');
    else { const r=S.reg; goLevel('dep'); S.reg=r; const l=findLayer(regLayer,r); if(l) map.fitBounds(l.getBounds(),{padding:[20,20]}); render(); }
  });
}
function render(){
  document.getElementById('yearOut').textContent=YEARS[S.yi];
  computeBreaks(allCodes(), source(S.lvl));
  const l=currentLayer(); if(l) l.setStyle(styleFor(()=>source(S.lvl)));
  legend(); top(); crumb(); showInfo();
}
let flashT;
function flash(t,sticky){const m=document.getElementById('msg'); clearTimeout(flashT);
  if(!t){m.style.display='none';return;} m.textContent=t; m.style.display='block';
  if(!sticky) flashT=setTimeout(()=>m.style.display='none',3500);}

regLayer.addTo(map);
render();
window.addEventListener('resize',()=>showInfo());
})();
</script>
</body>
</html>
"""
# --- TEMPLATE END ---

# ---------- Données Gold -> JSON ----------
reg_pd = spark.table("gold_delinquance_region").toPandas()
dep_pd = spark.table("gold_delinquance_departement").toPandas()
com_pd = spark.table("gold_delinquance_commune").toPandas()

years = sorted(int(y) for y in dep_pd["annee"].dropna().unique())
data = {
    "years": years,
    "indicators": INDICATEURS,
    "reg": pack(reg_pd, "code_region", years),
    "dep": pack(dep_pd, "code_departement", years),
    "com": pack(com_pd, "code_commune", years),
    "dep_region": dict(dep_pd[["code_departement", "code_region"]].drop_duplicates()
                       .astype(str).values.tolist()),
}

# ---------- Contours régions / départements ----------
reg_geo = slim_geojson(requests.get(GEO_BASE + "regions-avec-outre-mer.geojson", timeout=60).json())
dep_geo = slim_geojson(requests.get(GEO_BASE + "departements-avec-outre-mer.geojson", timeout=60).json())

os.makedirs(OUT_DIR, exist_ok=True)
html = build_html(data, reg_geo, dep_geo)
with open(f"{OUT_DIR}/carte_delinquance.html", "w", encoding="utf-8") as f:
    f.write(html)
print(f"Carte écrite : Files/gold/delinquance/carte_delinquance.html ({len(html)/1e6:.1f} Mo)")